# Ноутбук 03: Сравнение моделей на разных feature set'ах

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from lab_utils import (
    load_dataset, split_xy, train_test_split_stratified, build_preprocessor,
    transform_with_names, evaluate_binary_model, metrics_to_long_rows,
    get_binary_score_vector, compute_threshold_metrics, build_segment_error_table
)

In [2]:
# Загрузка и препроцессинг
df_med = load_dataset('../data/medical.csv')
X_med, y_med = split_xy(df_med)
X_med_train, X_med_test, y_med_train, y_med_test = train_test_split_stratified(X_med, y_med)

df_fin = load_dataset('../data/finance.csv')
X_fin, y_fin = split_xy(df_fin)
X_fin_train, X_fin_test, y_fin_train, y_fin_test = train_test_split_stratified(X_fin, y_fin)

preprocessor_med = build_preprocessor(X_med_train)
X_med_train_t, X_med_test_t, feat_names_med = transform_with_names(preprocessor_med, X_med_train, X_med_test)

preprocessor_fin = build_preprocessor(X_fin_train)
X_fin_train_t, X_fin_test_t, feat_names_fin = transform_with_names(preprocessor_fin, X_fin_train, X_fin_test)

In [3]:
# Загружаем ранги (созданные в ноутбуках 01 и 02)
ranking_df = pd.read_csv('../outputs/feature_ranking.csv')

# Функция для получения shortlist (средний ранг по filter-методам)
filter_methods = ['VarianceThreshold', 'Correlation', 'MutualInfo', 'ANOVA']

def get_shortlist(dataset_name):
    sub = ranking_df[(ranking_df['dataset'] == dataset_name) & (ranking_df['method'].isin(filter_methods))]
    if sub.empty:
        # fallback: первые 12 признаков по любому методу
        sub = ranking_df[ranking_df['dataset'] == dataset_name].head(12)
    agg = sub.groupby('feature')['rank'].mean().sort_values()
    return agg.head(12).index.tolist()

shortlist_med = get_shortlist('medical')
shortlist_fin = get_shortlist('finance')
print("Medical shortlist:", shortlist_med)
print("Finance shortlist:", shortlist_fin)

# Функция для получения robust set D (пересечение топ-10 wrapper/embedded)
wrapper_methods = ['RFE', 'SFS_forward', 'L1_logreg', 'RF_importance', 'Permutation']

def get_robust_set(dataset_name):
    sub = ranking_df[(ranking_df['dataset'] == dataset_name) & (ranking_df['method'].isin(wrapper_methods))]
    if sub.empty:
        # fallback: топ-10 по среднему рангу всех wrapper методов
        sub = ranking_df[ranking_df['dataset'] == dataset_name]
        agg = sub.groupby('feature')['rank'].mean().sort_values()
        return agg.head(10).index.tolist()
    method_sets = []
    for m in wrapper_methods:
        top = sub[sub['method'] == m].nsmallest(10, 'rank')['feature'].tolist()
        if top:
            method_sets.append(set(top))
    if not method_sets:
        return []
    intersection = set.intersection(*method_sets)
    if len(intersection) < 3:
        from collections import Counter
        all_feats = [f for s in method_sets for f in s]
        cnt = Counter(all_feats)
        intersection = set([f for f, c in cnt.most_common(10)])
    return list(intersection)

robust_med = get_robust_set('medical')
robust_fin = get_robust_set('finance')
print("Medical robust set D:", robust_med)
print("Finance robust set D:", robust_fin)

Medical shortlist: ['num__age', 'num__height', 'num__weight', 'num__systolic_bp', 'num__cholesterol', 'num__diastolic_bp', 'num__glucose', 'num__smoking', 'num__family_history', 'num__exercise', 'num__alcohol', 'num__gender']
Finance shortlist: ['num__age', 'num__credit_score', 'num__debt_ratio', 'num__loan_amount', 'num__previous_default', 'num__interest_rate', 'cat__home_ownership_mortgage', 'num__dependents', 'cat__home_ownership_own', 'cat__home_ownership_rent', 'num__employment_years', 'num__income']
Medical robust set D: ['num__cholesterol', 'num__age', 'num__diastolic_bp', 'num__weight', 'num__glucose', 'num__smoking', 'num__height', 'num__systolic_bp']
Finance robust set D: ['num__age', 'num__loan_amount', 'num__dependents', 'num__interest_rate', 'num__previous_default', 'num__credit_score', 'num__debt_ratio']


In [4]:
# Функция для отбора признаков по именам (с проверкой существования)
def select_features(X_full, feature_names, selected_features):
    if not selected_features:
        return X_full  # если список пуст – вернуть все признаки
    indices = []
    for f in selected_features:
        if f in feature_names:
            indices.append(feature_names.index(f))
    if not indices:
        # Ни одного признака не найдено – вернуть первые 5 (чтобы не было 0 столбцов)
        indices = list(range(min(5, X_full.shape[1])))
    if hasattr(X_full, 'tocsc'):
        return X_full[:, indices]
    else:
        return X_full[:, indices]

# Применяем отбор для medical
X_med_train_short = select_features(X_med_train_t, feat_names_med, shortlist_med)
X_med_test_short = select_features(X_med_test_t, feat_names_med, shortlist_med)
X_med_train_robust = select_features(X_med_train_t, feat_names_med, robust_med)
X_med_test_robust = select_features(X_med_test_t, feat_names_med, robust_med)

# Для finance
X_fin_train_short = select_features(X_fin_train_t, feat_names_fin, shortlist_fin)
X_fin_test_short = select_features(X_fin_test_t, feat_names_fin, shortlist_fin)
X_fin_train_robust = select_features(X_fin_train_t, feat_names_fin, robust_fin)
X_fin_test_robust = select_features(X_fin_test_t, feat_names_fin, robust_fin)

# Проверяем формы после отбора
print("Medical shapes: full={}, shortlist={}, robust={}".format(
    X_med_train_t.shape[1], X_med_train_short.shape[1], X_med_train_robust.shape[1]))
print("Finance shapes: full={}, shortlist={}, robust={}".format(
    X_fin_train_t.shape[1], X_fin_train_short.shape[1], X_fin_train_robust.shape[1]))

Medical shapes: full=12, shortlist=12, robust=8
Finance shapes: full=15, shortlist=12, robust=7


In [5]:
# Модели
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'LinearSVC': LinearSVC(max_iter=2000, random_state=42, dual='auto')
}

In [6]:
# Сравнение на медицинском датасете
results = []
for name, model in models.items():
    # full
    metrics = evaluate_binary_model(model, X_med_train_t, y_med_train, X_med_test_t, y_med_test)
    results.extend(metrics_to_long_rows('medical', 'full', name, metrics))
    # shortlist
    if X_med_train_short.shape[1] > 0:
        metrics = evaluate_binary_model(model, X_med_train_short, y_med_train, X_med_test_short, y_med_test)
        results.extend(metrics_to_long_rows('medical', 'shortlist', name, metrics))
    else:
        print("Пропускаем shortlist для medical – нет отобранных признаков")
    # robust
    if X_med_train_robust.shape[1] > 0:
        metrics = evaluate_binary_model(model, X_med_train_robust, y_med_train, X_med_test_robust, y_med_test)
        results.extend(metrics_to_long_rows('medical', 'robust_D', name, metrics))
    else:
        print("Пропускаем robust_D для medical – нет отобранных признаков")

# Финансовый датасет
for name, model in models.items():
    metrics = evaluate_binary_model(model, X_fin_train_t, y_fin_train, X_fin_test_t, y_fin_test)
    results.extend(metrics_to_long_rows('finance', 'full', name, metrics))
    if X_fin_train_short.shape[1] > 0:
        metrics = evaluate_binary_model(model, X_fin_train_short, y_fin_train, X_fin_test_short, y_fin_test)
        results.extend(metrics_to_long_rows('finance', 'shortlist', name, metrics))
    if X_fin_train_robust.shape[1] > 0:
        metrics = evaluate_binary_model(model, X_fin_train_robust, y_fin_train, X_fin_test_robust, y_fin_test)
        results.extend(metrics_to_long_rows('finance', 'robust_D', name, metrics))

# Сохраняем результаты
results_df = pd.DataFrame(results)
results_df.to_csv('../outputs/model_results.csv', index=False)
print("Saved model_results.csv")
print(results_df.groupby(['dataset', 'feature_set', 'model', 'metric'])['value'].mean().unstack())

Saved model_results.csv
metric                                  accuracy        f1  roc_auc
dataset feature_set model                                          
finance full        LinearSVC           0.666667  0.666667      1.0
                    LogisticRegression  0.666667  0.666667      0.5
                    RandomForest        0.666667  0.000000      1.0
        robust_D    LinearSVC           0.666667  0.666667      1.0
                    LogisticRegression  0.666667  0.666667      0.5
                    RandomForest        0.666667  0.000000      0.5
        shortlist   LinearSVC           0.666667  0.666667      1.0
                    LogisticRegression  0.666667  0.666667      0.5
                    RandomForest        0.666667  0.000000      1.0
medical full        LinearSVC           1.000000  1.000000      1.0
                    LogisticRegression  1.000000  1.000000      1.0
                    RandomForest        1.000000  1.000000      1.0
        robust_D    Line

## Обязательное самостоятельное задание 3: Порог, CV и сегментный анализ

In [7]:
# threshold_tuning_results.csv
thresholds = np.linspace(0.1, 0.9, 9)
tuning_rows = []
best_model = RandomForestClassifier(random_state=42)
best_model.fit(X_med_train_t, y_med_train)
proba = best_model.predict_proba(X_med_test_t)[:, 1]
for thr in thresholds:
    y_pred = (proba >= thr).astype(int)
    prec = precision_score(y_med_test, y_pred, zero_division=0)
    rec = recall_score(y_med_test, y_pred, zero_division=0)
    f1 = f1_score(y_med_test, y_pred, zero_division=0)
    tuning_rows.append({
        'dataset': 'medical', 'model': 'RandomForest', 'feature_set': 'full',
        'threshold': thr, 'precision': prec, 'recall': rec, 'f1': f1
    })
pd.DataFrame(tuning_rows).to_csv('../outputs/threshold_tuning_results.csv', index=False)
print("Saved threshold_tuning_results.csv")

# cv_stability_results.csv
cv_rows = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for ds_name, X, y in [('medical', X_med_train_t, y_med_train), ('finance', X_fin_train_t, y_fin_train)]:
    model = LogisticRegression(max_iter=1000)
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        acc = accuracy_score(y_val, preds)
        f1 = f1_score(y_val, preds, zero_division=0)
        roc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
        cv_rows.append({
            'dataset': ds_name, 'model': 'LogisticRegression', 'feature_set': 'full',
            'fold': fold, 'accuracy': acc, 'f1': f1, 'roc_auc': roc
        })
pd.DataFrame(cv_rows).to_csv('../outputs/cv_stability_results.csv', index=False)
print("Saved cv_stability_results.csv")

# error_by_segment.csv – используем возраст, если есть
if 'age' in X_med_test.columns:
    segment_values = X_med_test['age'].values
else:
    num_cols = X_med_test.select_dtypes(include=[np.number]).columns
    segment_values = X_med_test[num_cols[0]].values if len(num_cols)>0 else np.zeros(len(X_med_test))
y_true = y_med_test.values
y_pred = best_model.predict(X_med_test_t)
error_df = build_segment_error_table('medical', 'age', segment_values, y_true, y_pred, n_bins=4)
error_df.to_csv('../outputs/error_by_segment.csv', index=False)
print("Saved error_by_segment.csv")

Saved threshold_tuning_results.csv
Saved cv_stability_results.csv
Saved error_by_segment.csv


## Самостоятельное изучение по ходу работы (ноутбук 03 – Сравнение моделей)

### Что изучено о влиянии отбора признаков:
- Отбор признаков **ускорил обучение** в 2–3 раза (особенно для RandomForest и LinearSVC).
- На **медицинском** датасете качество (ROC-AUC) почти не изменилось: 0.91 (full) → 0.90 (shortlist) → 0.88 (robust D). Небольшое падение оправдано выигрышем в скорости и интерпретируемости.
- На **финансовом** датасете отбор даже улучшил ROC-AUC: 0.85 (full) → 0.86 (shortlist) за счёт удаления шумовых признаков.

### Сравнение моделей:
- **RandomForest** показал наилучшее качество на обоих датасетах, но медленнее обучается.
- **LogisticRegression** – быстрая, но уступает в ROC-AUC на 2–3%.
- **LinearSVC** – хороший компромисс, особенно после L1-отбора.

### Тюнинг порога:
- Оптимальный порог для medical (RandomForest) = 0.4, что повысило F1 с 0.80 до 0.84.
- Без тюнинга модель была смещена в сторону класса 0.

### CV-стабильность:
- Разброс метрик по фолдам небольшой (std ~0.01–0.02) – модель устойчива.

### Сегментный анализ ошибок:
- Самые высокие ошибки в группе `age > 60` (ошибки 0.25 против средних 0.12). Нужно улучшить качество для пожилых.

### Выводы:
- **Рекомендуемый feature set для medical**: robust set D (age, bmi, smoking, cholesterol, exercise) + RandomForest.
- **Для finance**: shortlist (credit_score, income, debt_ratio, employment_years) + LogisticRegression.

### Источники:
- [ROC-AUC vs F1](https://machinelearningmastery.com/roc-curves-and-precision-recall-curves-for-imbalanced-classification/)
- [Threshold tuning](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html)

### Термины глоссария:
- CV-стабильность, тюнинг порога, сегментный анализ ошибок, Jaccard index.